# Comparación de líneas base · NVIDIA A100 · versión optimizada

**Qué hace:** entrena y valida `deeplab_compacta` y `pspnet_compacta` para 5 semillas (10 corridas), guarda el entrenamiento por época, checkpoints de los mejores modelos y, **solo cuando todas las corridas están cerradas**, abre el test reservado y guarda sus resultados.

**Procedencia:** adaptación de `Experimento(1).ipynb` (mantiene la reconstrucción de los ZIP divididos). `Bases_v2.ipynb` usa exactamente el mismo código de modelos y protocolo; no son fases diferentes.

**Optimización:** caché persistente de imágenes/máscaras en memoria de GPU (si hay espacio, con reserva), eliminación de sincronizaciones innecesarias por paso, cuDNN benchmark y TF32 para A100. Se conserva **batch size = 1**, resolución 256, la pérdida BCE+Dice, optimizador Adam, épocas/semillas y conjunto de validación/test. **Los resultados numéricos pueden variar** al activar el perfil rápido (no se garantiza determinismo bit a bit); identifica este perfil como una nueva ejecución, no como continuación de checkpoints antiguos.

**Resultados:** se guardan localmente en `/content/IMO_resultados_A100`; hay respaldo comprimido automático en Google Drive después de cada corrida terminada y después del test. Si usas un runtime que no permite montar Drive, cambia `USAR_BACKUP_DRIVE = False` y descarga el ZIP manualmente antes de desconectar. La aceleración real depende de los datos y del entorno: **no se garantiza una duración concreta por época**.

**Preparación:** en Colab, *Entorno de ejecución → Cambiar tipo de entorno de ejecución → A100*. Sube la carpeta de partes ZIP (o los ZIP íntegros) y ejecuta las celdas en orden. El test permanece reservado hasta el final.


In [1]:
from pathlib import Path
import sys, hashlib, zipfile, json, csv
import torch
assert Path('/content').is_dir() and torch.cuda.is_available(), 'Se requiere un kernel GPU de Colab'
gpu = torch.cuda.get_device_name(0)
print('GPU detectada:', gpu)
if 'A100' not in gpu.upper():
    print('ATENCIÓN: este entorno no está usando una A100; las optimizaciones funcionan, pero el rendimiento será distinto.')
raiz=Path('/content/imo_v2')
salida=Path('/content/IMO_resultados_A100')
USAR_BACKUP_DRIVE = True  # Si no puedes montar Drive, cambia a False y descarga el ZIP final
RUTA_BACKUP = Path('/content/drive/MyDrive/Tesis_IMO_A100')
if USAR_BACKUP_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUTA_BACKUP.mkdir(parents=True, exist_ok=True)
    backup_zip = RUTA_BACKUP / 'IMO_resultados_A100.zip'
    if backup_zip.exists() and not salida.exists():
        salida.mkdir(parents=True)
        with zipfile.ZipFile(backup_zip) as z:
            destino = salida.resolve()
            assert all((salida / name).resolve().is_relative_to(destino) for name in z.namelist()), 'ZIP de respaldo con rutas no seguras'
            z.extractall(salida)
        print('Resultados anteriores restaurados desde Drive')

def extraer(nombre, huella):
    import time
    dst=Path('/content')/nombre
    if not dst.exists() or hashlib.sha256(dst.read_bytes()).hexdigest()!=huella:
        for intento in range(240):
            trozos=sorted(Path('/content').rglob(Path(nombre).stem+'_*.bin'))
            if sum(f.stat().st_size for f in trozos)=={'imo_colab_train_val.zip':184858955,'test_v2.zip':33081044}[nombre]:
                tmp=dst.with_suffix('.partial')
                with tmp.open('wb') as out:
                    for f in trozos: out.write(f.read_bytes())
                if hashlib.sha256(tmp.read_bytes()).hexdigest()==huella:
                    tmp.replace(dst)
                    break
            if intento % 6 == 0: print('Verificando transferencia:',nombre,len(trozos),flush=True)
            time.sleep(5)
        else: raise RuntimeError('Transferencia incompleta; verificar partes en /content')
    archivos=[p for p in {Path.cwd()/nombre,Path('/content')/nombre} if p.is_file()]
    assert archivos, f'Subir {nombre} con Upload to Colab'
    p=archivos[0]
    assert hashlib.sha256(p.read_bytes()).hexdigest()==huella, f'Paquete distinto: {nombre}'
    with zipfile.ZipFile(p) as z: z.extractall(raiz)
extraer('imo_colab_train_val.zip', '13045ae69c3cdfc60259d722624c258e57586737568df884e562fdacde42fedf')
scripts=raiz/'tesis_imo_para_codex/scripts'
sys.path.insert(0,str(scripts))


Mounted at /content/drive


In [2]:
codigo='"""Adaptaciones compactas para segmentación binaria, sin pesos externos."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom modelos_base import block\n\nclass Piramide(nn.Module):\n    def __init__(self, modo):\n        super().__init__()\n        self.encoder = nn.Sequential(block(3,16), nn.MaxPool2d(2), block(16,32),\n                                     nn.MaxPool2d(2), block(32,64))\n        self.modo = modo\n        if modo == \'aspp\':\n            self.ramas = nn.ModuleList([nn.Conv2d(64,16,1)] + [\n                nn.Conv2d(64,16,3,padding=d,dilation=d) for d in (2,4,8)] +\n                [nn.Sequential(nn.AdaptiveAvgPool2d(1),nn.Conv2d(64,16,1))])\n            canales = 80\n        else:\n            self.ramas = nn.ModuleList([nn.Sequential(nn.AdaptiveAvgPool2d(b),\n                           nn.Conv2d(64,16,1)) for b in (1,2,3,6)])\n            canales = 128\n        self.head = nn.Sequential(nn.Conv2d(canales,32,1),nn.ReLU(),nn.Conv2d(32,1,1))\n    def forward(self,x):\n        h = self.encoder(x)\n        partes = [h] if self.modo == \'psp\' else []\n        for rama in self.ramas:\n            y = F.relu(rama(h))\n            if y.shape[-2:] != h.shape[-2:]:\n                y = F.interpolate(y,size=h.shape[-2:],mode=\'bilinear\',align_corners=False)\n            partes.append(y)\n        return F.interpolate(self.head(torch.cat(partes,1)),size=x.shape[-2:],\n                             mode=\'bilinear\',align_corners=False)\nBASES = {\'deeplab_compacta\': lambda: Piramide(\'aspp\'), \'pspnet_compacta\': lambda: Piramide(\'psp\')}\n\n'
(scripts/'bases_v2.py').write_text(codigo, encoding='utf8')
from bases_v2 import BASES
runner_source='"""Piloto portable: train/val únicamente, checkpoints por época y semillas comunes; perfil A100.\n\nNo modifica particiones ni accede al test. Las adaptaciones no son réplicas de\nlos artículos. La aceleración con TF32/cuDNN benchmark no es determinista bit a bit.\nLas mismas semillas no garantizan identidad numérica entre GPU/versiones.\n"""\nimport hashlib\nimport json\nimport os\nimport random\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport PIL\nimport torch\nfrom dataset_ready import load_example, metrics, partition\nfrom modelos_base import MODELS\n\n\ndef environment():\n    return {\'torch\': str(torch.__version__), \'numpy\': np.__version__,\n            \'pillow\': PIL.__version__, \'cuda\': torch.version.cuda,\n            \'device\': torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'cpu\'}\n\n\ndef setup(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    torch.set_num_threads(2)\n    # Perfil A100: acelera kernels de convolución para tamaños fijos (256 x 256).\n    # NO garantiza resultados bit a bit idénticos al protocolo determinista T4.\n    torch.backends.cudnn.benchmark = True\n    torch.backends.cudnn.deterministic = False\n    torch.backends.cudnn.allow_tf32 = True\n    torch.backends.cuda.matmul.allow_tf32 = True\n    torch.use_deterministic_algorithms(False)\n\n\n# El conjunto train/val se lee una sola vez por imagen y se reutiliza durante\n# TODAS las épocas, modelos y semillas. El test solo se lee cuando se solicita.\n# La reserva de memoria protege de OOM en GPUs menores a la A100.\n_CPU_CACHE = {}\n_GPU_CACHE = {}\n_RESERVA_GPU_BYTES = 2 * 1024**3\n\n\ndef tensors(row, device):\n    key = row[\'id\']\n    if device.type == \'cuda\' and key in _GPU_CACHE:\n        return _GPU_CACHE[key]\n    if key not in _CPU_CACHE:\n        x, y, valid = load_example(row)\n        _CPU_CACHE[key] = (torch.from_numpy(x.copy())[None],\n                           torch.from_numpy(y.astype(np.float32))[None, None],\n                           torch.from_numpy(valid.copy())[None, None])\n    parts = _CPU_CACHE[key]\n    if device.type == \'cuda\':\n        required = sum(t.numel() * t.element_size() for t in parts)\n        free_bytes, _ = torch.cuda.mem_get_info(device)\n        if free_bytes > required + _RESERVA_GPU_BYTES:\n            _GPU_CACHE[key] = tuple(t.to(device, non_blocking=True) for t in parts)\n            del _CPU_CACHE[key]\n            return _GPU_CACHE[key]\n    return tuple(t.to(device) for t in parts)\n\n\ndef loss_fn(logits, target, valid):\n    if logits.shape != target.shape:\n        raise ValueError(\'La salida no coincide con la máscara\')\n    p, t = torch.sigmoid(logits)[valid], target[valid]\n    bce = torch.nn.functional.binary_cross_entropy_with_logits(logits[valid], t)\n    dice = 1 - (2 * (p*t).sum() + 1e-6) / (p.sum() + t.sum() + 1e-6)\n    return (bce + dice) / 2\n\n\ndef sync(device):\n    if device.type == \'cuda\':\n        torch.cuda.synchronize()\n\n\ndef smoke(output):\n    """Cinco pasos/modelo; prueba funcional, NO medición de calidad predictiva."""\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    rows = partition(\'train\')[:5]\n    results = []\n    for name, constructor in MODELS.items():\n        setup(11)\n        model = constructor().to(device)\n        optimizer = torch.optim.Adam(model.parameters(), lr=0.001)\n        if device.type == \'cuda\':\n            torch.cuda.reset_peak_memory_stats()\n        times = []\n        for row in rows:\n            sync(device); start = time.perf_counter()\n            x, y, v = tensors(row, device)\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(x), y, v)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Pérdida no finita\')\n            loss.backward(); optimizer.step()\n            sync(device); times.append(time.perf_counter() - start)\n        results.append({\'model\': name, \'steps\': len(times), \'seconds_per_step\': times,\n                        \'parameters\': sum(p.numel() for p in model.parameters()),\n                        \'peak_gpu_allocated_mib\': torch.cuda.max_memory_allocated()/2**20\n                        if device.type == \'cuda\' else None})\n        print(results[-1], flush=True)\n        del model, optimizer, x, y, v, loss\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n    report = {\'environment\': environment(), \'results\': results, \'seed\': 11,\n              \'quality_metrics\': \'No se evalúan: son cinco pasos de prueba\'}\n    (output/\'smoke.json\').write_text(json.dumps(report, indent=2), encoding=\'utf-8\')\n    return report\n\n\n@torch.no_grad()\ndef evaluate(model, rows, device, guard=None):\n    model.eval(); grouped = defaultdict(list); detail = []\n    for row in rows:\n        if guard:\n            guard()\n        x, y, v = tensors(row, device)\n        scores = metrics(torch.sigmoid(model(x))[0, 0].cpu().numpy(),\n                         y[0, 0].cpu().numpy(), v[0, 0].cpu().numpy())\n        grouped[row[\'group_id\']].append(scores[\'Dice\'])\n        detail.append({\'id\': row[\'id\'], \'group_id\': row[\'group_id\'], **scores})\n    return float(np.mean([np.mean(values) for values in grouped.values()])), detail\n\n\ndef run(name, seed, output, epochs=30, lr=0.001, patience=5,\n        identity=None, guard=None, max_epochs_this_call=None, force_cpu=False):\n    """Reanuda al final de la última época guardada; igual presupuesto/modelo.\n\n    Un corte dentro de una época repite esa época desde el checkpoint anterior.\n    Cargar checkpoints solo de esta ejecución privada: torch.load puede ejecutar\n    código cuando weights_only=False. No aceptar checkpoints de terceros.\n    """\n    if name not in MODELS or epochs < 1:\n        raise ValueError(\'Configuración inválida\')\n    setup(seed)\n    device = torch.device(\'cpu\' if force_cpu else (\'cuda\' if torch.cuda.is_available() else \'cpu\'))\n    root = Path(__file__).resolve().parents[1]\n    # La identidad incluye todos los archivos del paquete, por tanto código,\n    # particiones, imágenes y máscaras. Cambios rechazan una reanudación silenciosa.\n    package_digest = identity or hashlib.sha256((root.parent/\'package_manifest.json\').read_bytes()).hexdigest()\n    config = {\'model\': name, \'seed\': seed, \'epochs\': epochs, \'lr\': lr,\n              \'patience\': patience, \'batch_size\': 1, \'size\': 256,\n              \'package_manifest_sha256\': package_digest, \'environment\': environment(),\n              \'execution_device\': str(device),\n              \'performance_profile\': \'A100_cache_tf32_cudnn_benchmark_batch1\'}\n    dest = Path(output)/f\'{name}_seed{seed}_lr{lr:g}\'\n    dest.mkdir(parents=True, exist_ok=True)\n    cfg = dest/\'config.json\'\n    if cfg.exists() and json.loads(cfg.read_text()) != config:\n        raise ValueError(\'Cambió configuración, datos o entorno. Usar otra carpeta de resultados.\')\n    cfg.write_text(json.dumps(config, indent=2), encoding=\'utf-8\')\n    model = MODELS[name]().to(device)\n    optimizer = torch.optim.Adam(model.parameters(), lr=lr)\n    train, val = partition(\'train\'), partition(\'val\')\n    start_epoch, best, stale, history = 0, -1.0, 0, []\n    checkpoint = dest/\'last.pt\'\n    if checkpoint.exists():\n        saved = torch.load(checkpoint, map_location=\'cpu\', weights_only=False)\n        model.load_state_dict(saved[\'model\']); optimizer.load_state_dict(saved[\'optimizer\'])\n        start_epoch, best, stale, history = saved[\'epoch\']+1, saved[\'best\'], saved[\'stale\'], saved[\'history\']\n        torch.set_rng_state(saved[\'torch_rng\'])\n        if device.type == \'cuda\' and saved[\'cuda_rng\'] is not None:\n            torch.cuda.set_rng_state_all(saved[\'cuda_rng\'])\n    end_epoch = epochs if max_epochs_this_call is None else min(epochs, start_epoch+max_epochs_this_call)\n    for epoch in range(start_epoch, end_epoch):\n        if stale >= patience:\n            break\n        model.train(); order = list(train)\n        random.Random(seed * 100000 + epoch).shuffle(order)\n        sync(device); start = time.perf_counter()\n        total_loss = torch.zeros((), device=device)\n        for row in order:\n            if guard:\n                guard()\n            x, y, v = tensors(row, device)\n            optimizer.zero_grad(set_to_none=True)\n            loss = loss_fn(model(x), y, v)\n            loss.backward(); optimizer.step()\n            total_loss += loss.detach()\n        if not bool(torch.isfinite(total_loss)):\n            raise ValueError(\'Pérdida no finita. Se conserva el checkpoint de la época anterior.\')\n        score, details = evaluate(model, val, device, guard=guard)\n        improved = score > best\n        if improved:\n            best, stale = score, 0\n            torch.save(model.state_dict(), dest/\'best.tmp\')\n            os.replace(dest/\'best.tmp\', dest/\'best.pt\')\n            (dest/\'best_val_predictions.json\').write_text(json.dumps(details), encoding=\'utf-8\')\n        else:\n            stale += 1\n        sync(device)\n        history.append({\'epoch\': epoch+1, \'train_loss\': float((total_loss/len(train)).item()),\n                        \'val_group_Dice\': score, \'seconds\': time.perf_counter()-start})\n        torch.save({\'model\': model.state_dict(), \'optimizer\': optimizer.state_dict(),\n                    \'epoch\': epoch, \'best\': best, \'stale\': stale, \'history\': history,\n                    \'torch_rng\': torch.get_rng_state(),\n                    \'cuda_rng\': torch.cuda.get_rng_state_all() if device.type == \'cuda\' else None},\n                   dest/\'last.tmp\')\n        os.replace(dest/\'last.tmp\', checkpoint)\n        (dest/\'history.json\').write_text(json.dumps(history, indent=2), encoding=\'utf-8\')\n        print(name, seed, history[-1], flush=True)\n    result = {\'model\': name, \'seed\': seed, \'lr\': lr, \'best_val_group_Dice\': best,\n              \'epochs_completed\': len(history), \'test_evaluated\': False,\n              \'parameters\': sum(p.numel() for p in model.parameters()),\n              \'completed\': len(history) >= epochs or stale >= patience}\n    (dest/\'result.json\').write_text(json.dumps(result, indent=2), encoding=\'utf-8\')\n    return result\n'
(scripts/'colab_runner.py').write_text(runner_source, encoding='utf8')
sys.modules.pop('colab_runner',None)
from colab_runner import MODELS, run, evaluate, partition
MODELS.update(BASES)
seeds=[11,23,37,53,71]
identity=hashlib.sha256((raiz/'package_manifest.json').read_bytes()+codigo.encode()+runner_source.encode()).hexdigest()
import shutil

def respaldar_resultados():
    """Guarda un ZIP al completar cada corrida; no bloquea el test si falla Drive."""
    if not USAR_BACKUP_DRIVE:
        return
    archivo = Path(shutil.make_archive('/content/IMO_resultados_A100', 'zip', salida))
    temporal = RUTA_BACKUP/'IMO_resultados_A100.zip.part'
    shutil.copy2(archivo, temporal)
    temporal.replace(RUTA_BACKUP/'IMO_resultados_A100.zip')
    print('Respaldo en Drive:', RUTA_BACKUP/'IMO_resultados_A100.zip', flush=True)

for semilla in seeds:
    for red in BASES:
        r=run(red,semilla,salida,epochs=30,lr=0.001,patience=5,identity=identity)
        assert r['completed'], f'Corrida incompleta: {red}, semilla={semilla}'
        respaldar_resultados()

deeplab_compacta 11 {'epoch': 1, 'train_loss': 0.28273555636405945, 'val_group_Dice': 0.6650926272543246, 'seconds': 33.37127266899995}
deeplab_compacta 11 {'epoch': 2, 'train_loss': 0.19320979714393616, 'val_group_Dice': 0.7135478992398443, 'seconds': 6.731120393999959}
deeplab_compacta 11 {'epoch': 3, 'train_loss': 0.1674860715866089, 'val_group_Dice': 0.7270981781802625, 'seconds': 6.803564219000009}
deeplab_compacta 11 {'epoch': 4, 'train_loss': 0.162852481007576, 'val_group_Dice': 0.6949486410387887, 'seconds': 6.690889458000129}
deeplab_compacta 11 {'epoch': 5, 'train_loss': 0.143807053565979, 'val_group_Dice': 0.7524735774331864, 'seconds': 6.827427458999864}
deeplab_compacta 11 {'epoch': 6, 'train_loss': 0.1339157074689865, 'val_group_Dice': 0.7470605309444459, 'seconds': 6.711287285000026}
deeplab_compacta 11 {'epoch': 7, 'train_loss': 0.1292148381471634, 'val_group_Dice': 0.7614039967277911, 'seconds': 6.762852924999834}
deeplab_compacta 11 {'epoch': 8, 'train_loss': 0.118493

In [3]:
# Abrir el test reservado solo con las diez corridas cerradas.
for red in BASES:
    for s in seeds:
        assert json.loads((salida/f'{red}_seed{s}_lr0.001/result.json').read_text())['completed']
extraer('test_v2.zip', 'c324a4a69eed19f78413789a1d557f8a0880975462081ecf96926428b56c68a7')
prep=scripts.parent/'resultados/preentrenamiento'
freeze=json.loads((prep/'freeze_manifest.json').read_text())['artifacts']
for name in ['train.csv','val.csv','test.csv','normalization_train.json']:
    assert hashlib.sha256((prep/name).read_bytes()).hexdigest()==freeze[name]
rows=partition('test',allow_test=True)
assert len(rows)==218 and len({r['group_id'] for r in rows})==122
assert not {r['group_id'] for r in rows} & {r['group_id'] for part in ['train','val'] for r in partition(part)}
detalle=[]
resumen=[]
for red in BASES:
    for s in seeds:
        net=BASES[red]().cuda()
        ck=(salida/f'{red}_seed{s}_lr0.001/best.pt')
        net.load_state_dict(torch.load(ck,map_location='cuda',weights_only=True))
        score,detail=evaluate(net,rows,torch.device('cuda'))
        detalle.extend(dict(model=red,seed=s,**r) for r in detail)
        resumen.append({'modelo': red, 'semilla': s, 'Dice_test_por_grupo': score,
                        'n_imagenes_test': len(detail),
                        'n_grupos_test': len({d['group_id'] for d in detail})})
        print(red,s,score)
with (salida/'por_imagen.csv').open('w',newline='') as f:
    w=csv.DictWriter(f,fieldnames=detalle[0]); w.writeheader(); w.writerows(detalle)
with (salida/'resumen_test.csv').open('w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=resumen[0]); w.writeheader(); w.writerows(resumen)
(salida/'resumen_test.json').write_text(json.dumps(resumen, indent=2), encoding='utf-8')
print('Resultados locales en:', salida)
zip_final = shutil.make_archive('/content/IMO_resultados_A100','zip',salida)
print('ZIP final:', zip_final)
respaldar_resultados()



deeplab_compacta 11 0.8435460669639995
deeplab_compacta 23 0.8693429707920914
deeplab_compacta 37 0.8448962293081022
deeplab_compacta 53 0.8526994721228603
deeplab_compacta 71 0.8954584972031779
pspnet_compacta 11 0.8539969556189453
pspnet_compacta 23 0.871966041192561
pspnet_compacta 37 0.8651459080936845
pspnet_compacta 53 0.8686923442488561
pspnet_compacta 71 0.8735950013992475
Resultados locales en: /content/IMO_resultados_A100
ZIP final: /content/IMO_resultados_A100.zip
Respaldo en Drive: /content/drive/MyDrive/Tesis_IMO_A100/IMO_resultados_A100.zip
